# Dataset specification and preprocessing with pydantic `FileSpec` models

This is a rendered copy of [filespec.ipynb](https://github.com/scikit-hep/coffea/blob/master/binder/filespec.ipynb). You can optionally run it interactively on [binder at this link](https://mybinder.org/v2/gh/scikit-hep/coffea/master?filepath=binder%2Ffilespec.ipynb)

Before an analysis can run, coffea needs to know *which files* make up each dataset, *where each file's events live* (the `Events` TTree, an RNTuple, a Parquet dataset), *how to split each file into work units* (the `steps`), and *what the columns look like* (the awkward `form`). Historically this information was passed around as nested Python dictionaries — the "fileset" — which no code validated: a mistyped key, an out-of-order step range, or a missing `num_entries` surfaced only much later, deep inside a Dask graph.

Coffea's `dataset_tools` now describe this information with **[pydantic](https://docs.pydantic.dev/) models**. Instead of a bare `dict`, a dataset is a `DatasetSpec`; a file is a `ROOTFileSpec` or `ParquetFileSpec`; a whole fileset is a `DataGroupSpec`. Modelling the fileset this way buys several things a plain dictionary cannot:

- **Validation at construction time.** Step ranges must be sorted, non-overlapping, and clamped to `num_entries`; a ROOT file must carry an `object_path` while a Parquet file must not. Violations raise a `ValidationError` *where the mistake is made*, not three stages downstream.
- **Self-describing types.** `ROOTFileSpec` vs `CoffeaROOTFileSpec` (fully preprocessed) is a type distinction, so "is this fileset ready to run?" is answerable by the type system rather than by probing dictionary keys.
- **Round-trippable serialization.** `model_dump_json()` / `model_validate_json()` give a canonical on-disk form, with computed fields (`num_selected_entries`) and compressed awkward forms handled for you.
- **A dual-input contract.** Every entry point still accepts the **legacy plain-`dict` fileset** and returns a `dict` in that case, so existing code keeps working while new code gets the typed models.

This notebook is in two parts:

- **Part I — the dataset specification model**, as it exists in coffea `master`: the `FileSpec` type hierarchy, the `DataGroupSpec` fileset, `preprocess()` on the dask-awkward backend, the saved form, and the dataset-manipulation helpers.
- **Part II — experimental extensions** that build on Part I: switchable execution backends, RNTuple and Parquet preprocessing, **union forms by adding datasets**, **user metadata extraction during preprocessing**, and **mutable (resizable) steps**. These live on a development branch on top of the `master` pydantic/dask-awkward preprocessing; their APIs may still change.


In [1]:
import os

import awkward as ak
import rich
from pydantic import ValidationError

from coffea.dataset_tools import (
    # File-level specifications
    ROOTFileSpec,
    ParquetFileSpec,
    CoffeaROOTFileSpec,
    CoffeaROOTFileSpecOptional,
    # File collections and dataset/group specifications
    InputFiles,
    PreprocessedFiles,
    DatasetSpec,
    DataGroupSpec,
    # Preprocessing and dataset manipulation
    preprocess,
    max_chunks,
    max_files,
    slice_chunks,
    filter_files,
    # dict <-> model conversion layer
    ModelFactory,
)


def _find_samples():
    """Locate coffea's ``tests/samples`` directory regardless of where the notebook runs."""
    here = os.getcwd()
    candidates = [here] + [os.path.abspath(os.path.join(here, *([".."] * n))) for n in range(1, 4)]
    for base in candidates:
        for rel in (("tests", "samples"), ("coffea", "tests", "samples")):
            cand = os.path.join(base, *rel)
            if os.path.isdir(cand):
                return cand
    raise FileNotFoundError("Could not locate coffea's tests/samples directory")


SAMPLES = _find_samples()


def sample(name):
    """Full path to a file in coffea's ``tests/samples`` directory."""
    return os.path.join(SAMPLES, name)


print("Using sample files from:", SAMPLES)


Using sample files from: /Users/nmangane/servicex_claude/coffea/binder/coffea/tests/samples


## Part I — The dataset specification model

Everything in this part is available in coffea `master`.


### 1. From plain dictionaries to validated models

The legacy fileset is a nested dictionary: dataset name → `{"files": {filename: object_path}, "metadata": {...}}`. A `DataGroupSpec` accepts exactly that shape and turns it into validated models — so migrating an existing analysis is often just wrapping the dict you already have.


In [2]:
legacy_fileset = {
    "DYJets": {
        "files": {sample("nano_dy.root"): "Events"},
        "metadata": {"process": "DY", "xsec": 6077.22},
    },
    "Data": {
        "files": {sample("nano_dimuon.root"): "Events"},
        "metadata": {"process": "data"},
    },
}

fileset = DataGroupSpec(legacy_fileset)
rich.print(fileset)


DataGroupSpec  2 datasets
├── DYJets  root | 1 file  ◔ 0/1 preprocessed
│   ├── files
│   │   └── …vicex_claude/coffea/binder/coffea/tests/samples/nano_dy.root
│   ├── metadata
│   │   ├── process: 'DY'
│   │   └── xsec: 6077.22
│   └── columns
│       └── no form
└── Data  root | 1 file  ◔ 0/1 preprocessed
    ├── files
    │   └── …x_claude/coffea/binder/coffea/tests/samples/nano_dimuon.root
    ├── metadata
    │   └── process: 'data'
    └── columns
        └── no form

The dictionary is now a tree of typed objects: `DataGroupSpec` → `DatasetSpec` → an `InputFiles` collection → per-file `ROOTFileSpec`s. The `metadata` survived untouched, and the file format was inferred from the extension.

The payoff is validation. A plain dict will happily hold nonsense step ranges; a `ROOTFileSpec` rejects them at construction, naming the offending field.


In [3]:
# steps must be sorted, non-overlapping [start, stop] pairs — this raises immediately
try:
    ROOTFileSpec(object_path="Events", steps=[[0, 100], [50, 200]])
except ValidationError as err:
    print("ValidationError (overlapping steps):")
    print(err)


ValidationError (overlapping steps):
1 validation error for ROOTFileSpec
  Value error, steps: start of step 1 (50) is less than stop of previous step (100) [type=value_error, input_value={'object_path': 'Events',...: [[0, 100], [50, 200]]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error


### 2. File-level specifications and format detection

Each physical file is one spec. The format is a `Literal` tag baked into the type, and the two formats enforce different invariants:

- `ROOTFileSpec` **requires** an `object_path` (the TTree/RNTuple name) — a ROOT file without one is meaningless.
- `ParquetFileSpec` **forbids** `object_path` — a Parquet dataset has no such concept.

This is a place where the model encodes a rule that a dict simply cannot.


In [4]:
root_spec = ROOTFileSpec(object_path="Events", steps=[[0, 20], [20, 40]])
print("ROOT spec format:", root_spec.format)
print("num_selected_entries (computed):", root_spec.num_selected_entries)
rich.print(root_spec)

# A Parquet file carries no object_path; supplying one is a validation error.
parquet_spec = ParquetFileSpec()
print("\nParquet spec format:", parquet_spec.format)
try:
    ParquetFileSpec(object_path="Events")
except ValidationError as err:
    print("\nValidationError (parquet may not have object_path):")
    print(err)


ROOT spec format: root
num_selected_entries (computed): 40


ROOTFileSpec(
    object_path='Events',
    steps=[[0, 20], [20, 40]],
    num_entries=None,
    format='root',
    lfn=None,
    pfn=None,
    storage_options=None,
    metadata=None,
    experimental_field_bitset=None,
    num_selected_entries=40
)


Parquet spec format: parquet

ValidationError (parquet may not have object_path):
1 validation error for ParquetFileSpec
object_path
  Input should be None [type=none_required, input_value='Events', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/none_required


### 3. Partially- vs fully-known files

Before preprocessing, a file's `steps`, `num_entries`, and `uuid` are usually unknown; afterwards they are all filled in. The type hierarchy makes this difference explicit:

- `CoffeaROOTFileSpecOptional` — `num_entries`/`uuid` may be `None` (partially known).
- `CoffeaROOTFileSpec` — `steps`, `num_entries`, and `uuid` are all **required** (fully preprocessed).

A collection whose entries all carry complete information can be **promoted** to the concrete type; one missing information stays optional. `ModelFactory` performs this promotion.


In [5]:
partial = CoffeaROOTFileSpecOptional(object_path="Events", steps=[[0, 40]])
print("Partial spec — num_entries known?", partial.num_entries is not None)

# With num_entries and uuid supplied, the same data satisfies the concrete type.
concrete = CoffeaROOTFileSpec(
    object_path="Events",
    steps=[[0, 40]],
    num_entries=40,
    uuid="a1b2c3d4-0000-0000-0000-000000000000",
)
print("Concrete spec — fully known:", concrete.num_entries, concrete.uuid)
rich.print(concrete)


Partial spec — num_entries known? False
Concrete spec — fully known: 40 a1b2c3d4-0000-0000-0000-000000000000


CoffeaROOTFileSpec(
    object_path='Events',
    steps=[[0, 40]],
    num_entries=40,
    format='root',
    lfn=None,
    pfn=None,
    storage_options=None,
    metadata=None,
    experimental_field_bitset=None,
    uuid='a1b2c3d4-0000-0000-0000-000000000000',
    num_selected_entries=40
)

### 4. File collections and the dict-in / dict-out contract

Two dict-like collections wrap the per-file specs:

- `InputFiles` — filename → file spec, possibly incompletely known (pre-preprocessing).
- `PreprocessedFiles` — filename → fully-known spec.

Both behave like mappings and support set-style algebra (`+`, `-`), `limit_files`, `limit_steps`, and `filter_files`. Crucially, the public APIs accept **either** a legacy dict **or** the pydantic models and return the *matching* type — pass a dict, get a dict back; pass a model, get a model. That is what lets the new machinery drop into existing analyses unchanged.


In [6]:
inputs = InputFiles({
    sample("nano_dy.root"): "Events",
    sample("nano_dimuon.root"): "Events",
})
print("InputFiles is a mapping of", len(inputs), "files")
for name, spec in inputs.items():
    print(" ", os.path.basename(name), "->", spec.format, spec.object_path)


InputFiles is a mapping of 2 files
  nano_dy.root -> root Events
  nano_dimuon.root -> root Events


### 5. Metadata and JSON serialization

A `DatasetSpec` bundles a file collection with free-form `metadata` (cross sections, process labels, systematics knobs) and, once preprocessed, a compressed awkward `form`. The whole thing round-trips through JSON with `model_dump_json()` / `model_validate_json()`.

`DatasetSpec` defines equality on the *decoded* form, deliberately ignoring the raw compressed-form bytes: zlib output is not reproducible, so two specs describing the same data compare equal even if their compressed strings differ — a correctness subtlety you would have to implement by hand with dicts.


In [7]:
dataset = DatasetSpec(
    files={sample("nano_dy.root"): "Events"},
    metadata={"process": "DY", "xsec": 6077.22, "year": 2018},
)

as_json = dataset.model_dump_json()
restored = DatasetSpec.model_validate_json(as_json)

print("Round-trips through JSON equal:", restored == dataset)
print("Restored metadata:", restored.metadata)
print("JSON length (chars):", len(as_json))


Round-trips through JSON equal: True
Restored metadata: {'process': 'DY', 'xsec': 6077.22, 'year': 2018}
JSON length (chars): 382


### 6. Preprocessing with dask-awkward

`preprocess()` opens each file, reads `num_entries`, tiles it into `steps` of `step_size`, records the `uuid`, and (with `save_form=True`, the default) extracts and stores the awkward form. It returns **two** filesets:

- `available` — only the files that were successfully read (ready to run);
- `updated` — every input file, including any that could not be opened.

With `skip_bad_files=True`, unreadable files are recorded in `updated` but omitted from `available`, so a single dead replica does not sink the job. In `master` this runs on the **dask-awkward** backend, building a Dask graph over the per-file metadata.


In [8]:
to_preprocess = DataGroupSpec({
    "DYJets": {
        "files": {sample("nano_dy.root"): "Events"},
        "metadata": {"process": "DY"},
    },
    "Data": {
        "files": {
            sample("nano_dimuon.root"): "Events",
            "nonexistent_replica.root": "Events",  # simulate a dead file
        },
    },
})

available, updated = preprocess(
    to_preprocess,
    step_size=15,
    save_form=True,
    skip_bad_files=True,
)

print("available datasets:", list(available))
print("Data files available :", [os.path.basename(f) for f in available["Data"].files])
print("Data files (updated) :", [os.path.basename(f) for f in updated["Data"].files])
rich.print({k: v.model_dump(exclude="compressed_form") for k, v in available.items()})


available datasets: ['DYJets', 'Data']
Data files available : ['nano_dimuon.root']
Data files (updated) : ['nano_dimuon.root', 'nonexistent_replica.root']


{
    'DYJets': {
        'files': {
            '/Users/nmangane/servicex_claude/coffea/binder/coffea/tests/samples/nano_dy.root': {
                'object_path': 'Events',
                'steps': [[0, 14], [14, 28], [28, 40]],
                'num_entries': 40,
                'format': 'root',
                'lfn': None,
                'pfn': None,
                'metadata': None,
                'experimental_field_bitset': 
'7fffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffff
fffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffff
fffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffff
fffffffffffffffffffffffffffffff',
                'uuid': 'a9490124-3648-11ea-89e9-f5b55c90beef',
                'num_selected_entries': 40
            }
        },
        'metadata': {'process': 'DY'},
        'format': 'root',
        'did': None
    },
    'Data': {
        'files': {
            '/Users/nmangane/servicex_claude/coffea/binder/coffea/tests/samples/nano_dimuon.root': {
                'object_path': 'Events',
                'steps': [[0, 14], [14, 28], [28, 40]],
                'num_entries': 40,
                'format': 'root',
                'lfn': None,
                'pfn': None,
                'metadata': None,
                'experimental_field_bitset': 
'1fffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffff
fffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffff
fffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffff
fffffffffffffffffffffffffffffffffffffffffffffffffffffffff',
                'uuid': 'a210a3f8-3648-11ea-a29f-f5b55c90beef',
                'num_selected_entries': 40
            }
        },
        'metadata': {},
        'format': 'root',
        'did': None
    }
}

### 7. The saved form and typetracers

Each preprocessed dataset carries the union of its files' awkward forms, stored compressed. Decode it with the `.form` property. The form describes the file's *flat* branches (NanoAOD stores `Muon_pt`, `Muon_eta`, … rather than a nested `Muon` record — the nesting is applied later by `NanoAODSchema`). From the form you can build a **typetracer**: a zero-length array carrying the full type structure but no data, which is exactly what coffea uses to discover, ahead of time, which columns an analysis touches.


In [9]:
dy = available["DYJets"]
form = dy.form
print("Number of branches:", len(form.fields))
print("First 10 branches:", form.fields[:10])

muon_branches = [f for f in form.fields if f.startswith("Muon_")]
print("Muon_* branches:", muon_branches[:8])

typetracer = ak.typetracer.typetracer_from_form(form)
print("\nTypetracer built from the saved form (length is unknown by construction):")
print("  number of branches:", len(typetracer.fields))
print("  Muon_pt present:", "Muon_pt" in typetracer.fields)
print("  Muon_pt type:", typetracer["Muon_pt"].type)


Number of branches: 1499
First 10 branches: ['run', 'luminosityBlock', 'event', 'HTXS_Higgs_pt', 'HTXS_Higgs_y', 'HTXS_stage1_1_cat_pTjet25GeV', 'HTXS_stage1_1_cat_pTjet30GeV', 'HTXS_stage1_1_fine_cat_pTjet25GeV', 'HTXS_stage1_1_fine_cat_pTjet30GeV', 'HTXS_stage_0']
Muon_* branches: ['Muon_dxy', 'Muon_dxyErr', 'Muon_dz', 'Muon_dzErr', 'Muon_eta', 'Muon_ip3d', 'Muon_jetPtRelv2', 'Muon_jetRelIso']

Typetracer built from the saved form (length is unknown by construction):
  number of branches: 1499
  Muon_pt present: True
  Muon_pt type: ## * [var * float32, parameters={"__doc__": "pt", "typename": "float[]"}]


### 8. Manipulating filesets

Once preprocessed, the fileset is easy to subset for quick tests or scale-outs. These helpers all take and return a `DataGroupSpec` (or the equivalent dict):

- `max_files` — keep at most *N* files per dataset;
- `max_chunks` / `slice_chunks` — keep at most *N* / a slice of the work chunks;
- `filter_files` — keep files matching a predicate.


In [10]:
small = max_chunks(available, 1)
print("After max_chunks(1):")
for name, ds in small.items():
    nsteps = sum(len(fs.steps) for fs in ds.files.values())
    print(f"  {name}: {len(ds.files)} file(s), {nsteps} chunk(s)")

# filter_files takes a (filename, filespec) predicate
only_dy = filter_files(available, lambda item: "nano_dy" in item[0])
print("\nfilter_files keeping only nano_dy*:")
for name, ds in only_dy.items():
    print(f"  {name}: {[os.path.basename(f) for f in ds.files]}")


After max_chunks(1):
  DYJets: 1 file(s), 1 chunk(s)
  Data: 1 file(s), 1 chunk(s)

filter_files keeping only nano_dy*:
  DYJets: ['nano_dy.root']
  Data: []


## Part II — Experimental extensions

The remaining sections build on Part I. They live on a development branch layered over `master`'s pydantic/dask-awkward preprocessing, and their APIs may still change. The pydantic models are what make these extensions composable: each one is either a new field on a spec, a new keyword to `preprocess()`, or a new operation on `DatasetSpec` — all of them serialize, validate, and round-trip alongside the core model without bespoke plumbing.


### 9. Switchable execution backends

`master` computes steps through dask-awkward. The experimental branch factors execution behind a `PreprocessBackend`, so the *same* `preprocess()` call can run on:

- `"dask"` — the default dask-awkward graph;
- `"iterative"` — immediate, synchronous, dask-free (handy for debugging and small jobs);
- `"futures"` — a dask-free `concurrent.futures` thread pool (`FuturesBackend(workers=...)`).

Because the result is a validated `DataGroupSpec`, the backends are trivially checked for agreement with `==` (which compares decoded forms, not compressed bytes).


In [11]:
from coffea.dataset_tools import FuturesBackend

root_fileset = DataGroupSpec({"DYJets": {"files": {sample("nano_dy.root"): "Events"}}})

avail_iterative, _ = preprocess(root_fileset, step_size=10, save_form=True, backend="iterative")
avail_futures, _ = preprocess(root_fileset, step_size=10, save_form=True, backend=FuturesBackend(workers=2))

print("iterative and futures backends agree:", avail_iterative == avail_futures)
rich.print({k: v.model_dump(exclude="compressed_form") for k, v in avail_iterative.items()})


iterative and futures backends agree: True


{
    'DYJets': {
        'files': {
            '/Users/nmangane/servicex_claude/coffea/binder/coffea/tests/samples/nano_dy.root': {
                'object_path': 'Events',
                'steps': [[0, 10], [10, 20], [20, 30], [30, 40]],
                'num_entries': 40,
                'format': 'root',
                'lfn': None,
                'pfn': None,
                'metadata': None,
                'experimental_field_bitset': 
'7fffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffff
fffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffff
fffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffff
fffffffffffffffffffffffffffffff',
                'uuid': 'a9490124-3648-11ea-89e9-f5b55c90beef',
                'num_selected_entries': 40
            }
        },
        'metadata': {},
        'format': 'root',
        'did': None
    }
}

### 10. RNTuple preprocessing

ROOT's RNTuple is the successor columnar container to the TTree. The experimental backend auto-detects TTree vs RNTuple per object, and can extract the awkward form **without** building a Dask graph. `preprocess_rntuple()` additionally *requires* every object to be an RNTuple (a TTree raises), which is useful as a guard for RNTuple-only productions.


In [12]:
from coffea.dataset_tools import preprocess_rntuple

rntuple_fileset = DataGroupSpec({"DYJets": {"files": {sample("nano_dy_rntuple.root"): "Events"}}})
rntuple_available, _ = preprocess_rntuple(rntuple_fileset, step_size=10, save_form=True, backend="iterative")

rich.print({k: v.model_dump(exclude="compressed_form") for k, v in rntuple_available.items()})


{
    'DYJets': {
        'files': {
            '/Users/nmangane/servicex_claude/coffea/binder/coffea/tests/samples/nano_dy_rntuple.root': {
                'object_path': 'Events',
                'steps': [[0, 10], [10, 20], [20, 30], [30, 40]],
                'num_entries': 40,
                'format': 'root',
                'lfn': None,
                'pfn': None,
                'metadata': None,
                'experimental_field_bitset': 
'7fffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffff
fffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffff
fffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffff
fffffffffffffffffffffffffffffff',
                'uuid': '5aa41eba-a9f5-11f0-ba63-0f00a8c0beef',
                'num_selected_entries': 40
            }
        },
        'metadata': {},
        'format': 'root',
        'did': None
    }
}

### 11. Parquet preprocessing

Parquet datasets preprocess through the same entry point. Parquet files carry no `object_path`, so the files map to `None`; steps can optionally follow the file's row groups.


In [13]:
from coffea.dataset_tools import preprocess_parquet

parquet_fileset = DataGroupSpec({"DYJets": {"files": {sample("nano_dy.parquet"): None}}})
parquet_available, _ = preprocess_parquet(parquet_fileset, step_size=10, save_form=True, backend="iterative")

rich.print({k: v.model_dump(exclude="compressed_form") for k, v in parquet_available.items()})


{
    'DYJets': {
        'files': {
            '/Users/nmangane/servicex_claude/coffea/binder/coffea/tests/samples/nano_dy.parquet': {
                'object_path': None,
                'steps': [[0, 10], [10, 20], [20, 30], [30, 40]],
                'num_entries': 40,
                'format': 'parquet',
                'lfn': None,
                'pfn': None,
                'metadata': None,
                'experimental_field_bitset': 
'7fffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffff
fffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffff
fffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffff
fffffffffffffffffffffffffffffff',
                'uuid': '24be1d49dc0e1cc4ae4a0ee765e3d17c27662d075bf923aeaade5fbec7e29fca',
                'num_selected_entries': 40
            }
        },
        'metadata': {},
        'format': 'parquet',
        'did': None
    }
}

### 12. Union forms by adding datasets

A dataset's saved form must describe **every** field that appears in **any** of its files. CMS NanoAOD makes this concrete: `GenModel_*` signal-scan flags, or `HLT_*` trigger bits, are frequently present in only some files of a dataset. `preprocess()` already unions the forms of the files *within* one `preprocess()` call. The experimental extension goes further: **adding two separately-preprocessed `DatasetSpec`s computes the union of their forms**, so datasets processed at different times or places can be combined *without reopening any files*.

The two sample files below are NanoAOD-like: each carries a different subset of `GenModel_TChiZH_*` model-point flags.


In [14]:
gen_a = sample("nano_genmodel_with20_700_1_with0_1100_200_with0_950_400.root")
gen_b = sample("nano_genmodel_without_700_1_with0_1100_200_with20_950_400.root")

grp_a, _ = preprocess(DataGroupSpec({"Signal": {"files": {gen_a: "Events"}}}),
                      save_form=True, backend="iterative", step_size=10)
grp_b, _ = preprocess(DataGroupSpec({"Signal": {"files": {gen_b: "Events"}}}),
                      save_form=True, backend="iterative", step_size=10)

ds_a, ds_b = grp_a["Signal"], grp_b["Signal"]
fields_a, fields_b = set(ds_a.form.fields), set(ds_b.form.fields)
print("fields only in A:", len(fields_a - fields_b))
print("fields only in B:", len(fields_b - fields_a))


fields only in A: 10
fields only in B: 93


Adding the two specs (`ds_a + ds_b`) merges their files and forms the **union** of their fields. Every field present in either operand appears in the combined form.


In [15]:
combined = ds_a + ds_b
print("fields in A :", len(fields_a))
print("fields in B :", len(fields_b))
print("fields in union:", len(combined.form.fields))
assert set(combined.form.fields) == (fields_a | fields_b)
print("union == A ∪ B:", set(combined.form.fields) == (fields_a | fields_b))


fields in A :

 804
fields in B : 887
fields in union: 897
union == A ∪ B: True


Each file in the combined dataset records, as an experimental **field bitset**, exactly which of the union's top-level fields it actually contains. This lets a reader know per file whether a `GenModel_*` branch is really present (versus supplied as a masked/`None` column by the union form). Decode it with `decode_field_bitset`.


In [16]:
from coffea.dataset_tools.forms import decode_field_bitset

union_fields = list(combined.form.fields)
for name, fs in combined.files.items():
    present = decode_field_bitset(fs.experimental_field_bitset, union_fields)
    genmodel = sorted(f for f in present if f.startswith("GenModel_TChiZH_7"))
    print(os.path.basename(name)[:42], "-> GenModel_TChiZH_7* present:", genmodel)


nano_genmodel_with20_700_1_with0_1100_200_ -> GenModel_TChiZH_7* present: ['GenModel_TChiZH_700_1', 'GenModel_TChiZH_700_100', 'GenModel_TChiZH_700_150', 'GenModel_TChiZH_700_200', 'GenModel_TChiZH_700_250', 'GenModel_TChiZH_700_300', 'GenModel_TChiZH_700_350', 'GenModel_TChiZH_700_450', 'GenModel_TChiZH_700_50', 'GenModel_TChiZH_700_500', 'GenModel_TChiZH_700_550', 'GenModel_TChiZH_700_574', 'GenModel_TChiZH_750_1', 'GenModel_TChiZH_750_100', 'GenModel_TChiZH_750_150', 'GenModel_TChiZH_750_200', 'GenModel_TChiZH_750_250', 'GenModel_TChiZH_750_300', 'GenModel_TChiZH_750_350', 'GenModel_TChiZH_750_550', 'GenModel_TChiZH_750_600']
nano_genmodel_without_700_1_with0_1100_200 -> GenModel_TChiZH_7* present: ['GenModel_TChiZH_700_100', 'GenModel_TChiZH_700_150', 'GenModel_TChiZH_700_200', 'GenModel_TChiZH_700_250', 'GenModel_TChiZH_700_300', 'GenModel_TChiZH_700_350', 'GenModel_TChiZH_700_400', 'GenModel_TChiZH_700_450', 'GenModel_TChiZH_700_50', 'GenModel_TChiZH_700_500', 'GenModel_TChiZH_70

Field *order* in a union form depends on the order the datasets were added, though form *equality* does not. For a byte-stable serialization, `union_with(other, sort_fields=True)` (or `canonicalize_form()` on an existing spec) sorts the record fields recursively so the JSON is reproducible regardless of merge order.


In [17]:
ab = ds_a.union_with(ds_b, sort_fields=True)
ba = ds_b.union_with(ds_a, sort_fields=True)
print("sorted union is order-independent (byte-stable form):",
      ab.model_dump()["compressed_form"] != None and list(ab.form.fields) == list(ba.form.fields))
print("fields are sorted:", list(ab.form.fields) == sorted(ab.form.fields))


sorted union is order-independent (byte-stable form): True
fields are sorted: True


Removing files runs the process in reverse: a `DatasetSpec`'s own `filter_files` / `limit_files` **prune** the union form down to the fields the surviving files actually carry (using the per-file bitsets), so a subset does not drag along columns none of its files provide.


In [18]:
# Keep only file A (its name contains "with20_700_1") and watch the form shrink.
only_a = combined.filter_files(filter_name=".*with20_700_1.*")
print("files kept:", [os.path.basename(f) for f in only_a.files])
print("union fields:", len(combined.form.fields), "-> pruned to:", len(only_a.form.fields))
print("pruned form matches file A's own fields:", set(only_a.form.fields) == fields_a)


files kept: ['nano_genmodel_with20_700_1_with0_1100_200_with0_950_400.root']
union fields: 897 -> pruned to: 804
pruned form matches file A's own fields: True


### 13. User metadata extraction during preprocessing

Preprocessing already opens every file, which is the natural moment to harvest per-file bookkeeping — sum-of-weights, run/lumi coverage, provenance — that an analysis needs later. Two hooks make this a first-class part of `preprocess()`:

- `metadata_extractor(file_handle) -> dict` runs once per file on the open handle, and its (JSON-serializable) result is stored as that file spec's `metadata`. It runs inside the per-file error handling, so an extraction failure participates in `skip_bad_files`.
- `metadata_reducer({filename: extracted}) -> dict` runs once per dataset over the extracted values, and its result merges into the *dataset's* `metadata`.

Because these ride on the pydantic models, the harvested metadata serializes and round-trips with the rest of the spec for free.


In [19]:
def n_branches(file_handle):
    """Per-file extractor: how many branches the Events tree has."""
    return {"n_branches": len(file_handle["Events"].keys())}


def total_branches(per_file):
    """Dataset-level reducer: sum the per-file branch counts."""
    return {"total_branches": sum(m["n_branches"] for m in per_file.values())}


meta_fileset = DataGroupSpec({
    "DYJets": {
        "files": {sample("nano_dy.root"): "Events", sample("nano_dimuon.root"): "Events"},
        "metadata": {"process": "DY"},
    }
})

meta_available, _ = preprocess(
    meta_fileset,
    save_form=False,
    backend="iterative",
    metadata_extractor=n_branches,
    metadata_reducer=total_branches,
)

ds = meta_available["DYJets"]
print("Dataset metadata (reducer merged in):", ds.metadata)
for name, fs in ds.files.items():
    print(" ", os.path.basename(name), "->", fs.metadata)


Dataset metadata (reducer merged in): {'process': 'DY', 'total_branches': 3100}
  nano_dy.root -> {'n_branches': 1499}
  nano_dimuon.root -> {'n_branches': 1601}


The per-file `metadata` lives only on the pydantic models — the legacy `dict` fileset format has nowhere to put it, so dict-in/dict-out conversions drop it. This is one more capability the typed models unlock over bare dictionaries.


### 14. Mutable (resizable) steps

The `steps` stored on a file spec are a *static* tiling chosen at preprocess time. But a scheduler watching real resource usage — say, a worker nearing its memory limit — can do better by **shrinking or growing subsequent chunks while a file is being processed**. The experimental `coffea.dataset_tools.mutable_steps` module models this as a `send`-channel generator: iterating yields `[start, stop]` steps, and `generator.send(new_size)` renegotiates the size of all *subsequent* steps, re-tiling the remaining entries as evenly as possible.

A resize can never produce a larger-than-requested step, and the remaining steps stay near-uniform:


In [20]:
from coffea.dataset_tools import mutable_steps

# Tiling [0, 1000) in steps of 250, then shrinking to 100 after the first step.
gen = mutable_steps.resizable_steps(0, 1000, 250)
first = next(gen)          # [0, 250]
after_resize = gen.send(100)   # renegotiate: subsequent steps <= 100
rest = list(gen)
print("first step (size 250):", first)
print("first step after send(100):", after_resize)
print("remaining steps:", rest)


first step (size 250): [0, 250]
first step after send(100): [250, 344]
remaining steps: [[344, 438], [438, 532], [532, 626], [626, 720], [720, 814], [814, 907], [907, 1000]]


The same channel works over a preprocessed file spec via `iter_file_steps`, tiling the file's covered regions and honoring resize requests across region and file boundaries.


In [21]:
spec = next(iter(avail_iterative["DYJets"].files.values()))
print("file covers num_entries =", spec.num_entries)

gen = mutable_steps.iter_file_steps(spec, step_size=15)
steps = [next(gen)]
steps.append(gen.send(5))  # after the first chunk, shrink to <= 5 entries
steps.extend(gen)
print("steps (start 15, resized to 5):", steps)


file covers num_entries = 40
steps (start 15, resized to 5): [[0, 14], [14, 19], [19, 24], [24, 28], [28, 32], [32, 36], [36, 40]]


A toy driver, `run_adaptive_steps`, closes the loop: it processes a dataset step by step and feeds each step's measured wall time to a resize policy. `WallTimeStepPolicy` targets a fixed wall time per step, clamped and damped so a single fast outlier cannot balloon the chunk size. The clock is injectable, so the control loop is demonstrable without real waiting.


In [22]:
from coffea.dataset_tools.mutable_steps import WallTimeStepPolicy, run_adaptive_steps

# A fake clock advancing a fixed 0.3s per reading; each step then "takes" 0.3s, which is
# over the 0.2s target, so the policy keeps shrinking the chunk size toward min_step_size.
_clock = {"t": 0.0}
def fake_clock():
    _clock["t"] += 0.3
    return _clock["t"]

def do_work(filename, step):
    return step[1] - step[0]  # pretend to process; return the chunk size

dataset = avail_iterative["DYJets"]
policy = WallTimeStepPolicy(target_seconds=0.2, min_step_size=4, max_step_size=20)
run = run_adaptive_steps(dataset, do_work, step_size=15, policy=policy, clock=fake_clock)

print("chunk sizes chosen as each step ran over the wall-time target:", run.step_sizes)


chunk sizes chosen as each step ran over the wall-time target: [14, 9, 6, 4, 4, 3]


### 15. Further experimental utilities

Several more conveniences ride on the same pydantic models. They are shown here as reference snippets rather than executed, since they pull in optional dependencies (`universal_pathlib`, `servicex`, ROOT):

**Rich display** — every spec renders as a formatted panel in a terminal (`rich.print(spec)`, used throughout this notebook) and as an HTML table in Jupyter, with no extra code.

**Credentialed remote handles via `universal_pathlib`** — a file spec can carry `storage_options` (endpoints, bearer tokens) that are *excluded from serialization* by default, so secrets are re-supplied at open time rather than persisted:

```python
spec = ROOTFileSpec(object_path="Events", storage_options={"token": "..."})
handle = spec.open("root://xrootd.example//store/data/file.root")  # opens via fsspec
upath = spec.upath("root://xrootd.example//store/data/file.root")  # a universal_pathlib.UPath
spec.close()
```

**ServiceX and RDataFrame converters** — a `DatasetSpec` maps to and from a ServiceX `Sample`/spec and an RDataFrame dataset spec:

```python
from coffea.dataset_tools import to_servicex_spec, from_servicex, to_rdf_spec, from_rdf_spec
sx_spec = to_servicex_spec(dataset_spec)
rdf_spec = to_rdf_spec(dataset_spec)
```

These keep the validated `DatasetSpec` as the single source of truth while interoperating with the wider ecosystem.


### Summary

Modelling the fileset with pydantic turns a pile of untyped dictionaries into a validated, serializable, self-describing object tree. Part I — the `FileSpec` hierarchy, `DataGroupSpec`, `preprocess()`, the saved form, and the manipulation helpers — is in coffea `master` today, and accepts the legacy dict format so existing analyses migrate incrementally. Part II shows how much that foundation enables: swappable execution backends, RNTuple and Parquet support, union forms that combine independently-preprocessed datasets, per-file metadata harvested during preprocessing, and mutable steps that a resource-aware scheduler can renegotiate mid-flight — each one a small, composable addition to the same model rather than a parallel code path.
